# 01 — Generate the batches

Four deterministic loads, no file and no customer data. This notebook is optional: the proof notebook rebuilds the same batches itself. Both copies are the same shape and have to stay that way.

| Batch | Rows | What it does |
| --- | --- | --- |
| `good` | 1000 | Replaces the table with 200 orders of 5 lines, unique `(order_id, line_no)` |
| `duplicate_replay` | 1000 | Inserts those same keys again with a later `load_ts` |
| `empty` | 0 | Deletes `bt_orders` and still writes a ledger row |
| `spike` | 10000 | Replaces the table with 10000 rows, same 5-line shape |

The sizes are fixed. There is no row-count argument.

`line_no` cycles 1 through 5, so `order_id` repeats inside a good batch. `DUPLICATE_COUNT(order_id)` is therefore not the pair check. `load_ts` is not part of the key, so replaying the same lines with a new timestamp is a duplicate of the pair.

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

## Reset

The counter goes back to 0. Batch ids are monotonic within a run, not across runs, and the latest attempt is still `MAX(batch_id)` on the ledger.

In [ ]:
BEGIN;
DELETE FROM bt_orders;
DELETE FROM bt_load_ledger;
UPDATE bt_batch_id SET id = 0;
COMMIT;

## good

200 orders, lines 1 through 5, unique `(order_id, line_no)`.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1,
  MOD(ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, 5) + 1,
  'SKU-' || LPAD(TO_VARCHAR(FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1), 6, '0'),
  1,
  CURRENT_TIMESTAMP(),
  (SELECT id FROM bt_batch_id)
FROM TABLE(GENERATOR(ROWCOUNT => 1000));
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'good', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 1000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

## duplicate_replay

Same keys as the good batch, inserted again with a `load_ts` five minutes later. Run this after `good` and before `empty`.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT order_id, line_no, sku, qty,
       DATEADD('minute', 5, CURRENT_TIMESTAMP()),
       (SELECT id FROM bt_batch_id)
FROM bt_orders
WHERE batch_id = (SELECT MIN(batch_id) FROM bt_orders);
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'duplicate_replay', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 1000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

## empty

Deletes the table and still writes a ledger row. This is the case that makes the ledger the source of "latest attempt": `MAX(batch_id)` on `bt_orders` would skip it.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'empty', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 0, 'synthetic load'
FROM bt_batch_id;
COMMIT;

## spike

Replaces the table with 10000 rows, the same 5-line key shape.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1,
  MOD(ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, 5) + 1,
  'SKU-' || LPAD(TO_VARCHAR(FLOOR((ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) / 5) + 1), 6, '0'),
  1,
  CURRENT_TIMESTAMP(),
  (SELECT id FROM bt_batch_id)
FROM TABLE(GENERATOR(ROWCOUNT => 10000));
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'spike', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 10000, 'synthetic load'
FROM bt_batch_id;
COMMIT;

In [ ]:
SELECT batch_id, batch_name, row_count, note
FROM bt_load_ledger
ORDER BY batch_id;